# Sahayak - re-evaluate the ORBIT checkpoints after the evaluation fix (NO training)
Before you start: in Google Drive create the folder `MyDrive/sahayak_orbit_out/checkpoints` and upload into it the files
`embedder_C_partial_open.pt`, `embedder_C_partial_open_xv.pt`, `embedder_D_lora_open.pt` (downloaded earlier from the Kaggle output).
Then **Runtime -> Change runtime type -> T4 GPU** and **Runtime -> Run all**. About 1.5 hours; keep the tab open.
Results go to Drive `MyDrive/sahayak_orbit_out/reports_fixed` (the correct "ours" rows on the test split).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, zipfile, re, shutil, requests, glob, json
OUT = '/content/drive/MyDrive/sahayak_orbit_out'; FIX = OUT + '/reports_fixed'; os.makedirs(FIX, exist_ok=True)
os.chdir('/content')
!rm -rf sahayak2
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git sahayak2
os.chdir('/content/sahayak2')
!pip -q install timm scikit-learn tqdm pyyaml
!nvidia-smi | head -12
os.makedirs('reports', exist_ok=True); os.makedirs('checkpoints', exist_ok=True)
shutil.copytree(f'{OUT}/checkpoints', 'checkpoints', dirs_exist_ok=True)
print(sorted(os.listdir('checkpoints')))
assert os.path.exists('checkpoints/embedder_C_partial_open_xv.pt'), 'upload the checkpoints to Drive first (see the top of this notebook)'
RAW, ROOT = '/content/orbit_raw', '/content/orbit_instances'
EV = f'--root {ROOT} --split test --episodes 200 --k-shots 1 3 5 10 --cross-video'
def sh(cmd): get_ipython().system(cmd)
def save(): shutil.copytree('reports', FIX, dirs_exist_ok=True)


In [ ]:
files = requests.get('https://api.figshare.com/v2/articles/14294597', timeout=60).json()['files']
byname = {f['name']: f for f in files}
os.makedirs(RAW, exist_ok=True)
EVERY = 10
def fetch(name):
    f = byname[name]; z = '/content/' + name
    print('downloading', name, round(f['size'] / 1e9, 1), 'GB ...', flush=True)
    subprocess.run(['wget', '-q', '-c', '-O', z, f['download_url']], check=True)
    kept = total = 0
    pat = re.compile(r'(\d+)\.(?:jpe?g|png)$', re.I)
    with zipfile.ZipFile(z) as zf:
        for m in zf.infolist():
            if m.is_dir():
                continue
            total += 1
            k = pat.search(m.filename)
            if k and int(k.group(1)) % EVERY != 0:
                continue
            zf.extract(m, RAW); kept += 1
    os.remove(z)
    print(f'{name}: kept {kept} of {total} frames', flush=True)
fetch('validation_224.zip'); fetch('train_224.zip')
sh(f'python tools/prepare_orbit.py --src {RAW} --out {ROOT} --every 1 2>&1 | tail -5')
shutil.rmtree(RAW, ignore_errors=True)
n = len(os.listdir(ROOT)); print(n, 'object folders (expect about 330)'); assert n > 100


In [ ]:
# NOTE: the split is re-created from the folder names with the same seed, so it equals the earlier split (214/59/55). Check:
sh(f"python -c \"import json; s=json.load(open('{ROOT}/splits.json')); print({{k: len(v) for k, v in s.items()}})\"")


In [ ]:
def ev(tag, extra=''):
    sh(f'python -m src.eval.episodic {EV} {extra} --tag {tag} 2>&1 | tee reports/log_eval_{tag}.txt | tail -45')
    save()
ev('C_partial_open_xv_fixed', '--ckpt checkpoints/embedder_C_partial_open_xv.pt')
ev('C_partial_open_fixed', '--ckpt checkpoints/embedder_C_partial_open.pt')
ev('B_dinov2_frozen_fixed')
if os.path.exists('checkpoints/embedder_D_lora_open.pt'):
    ev('D_lora_open_fixed', '--ckpt checkpoints/embedder_D_lora_open.pt')
print('Saved to Drive:', FIX)
